# 01 Recopilacion y transformacion de datos

Este notebook desarrolla la **Fase 1** del proyecto: recopilacion, limpieza y transformacion del dataset de indicadores ENSO (El Nino / La Nina) para el periodo 1950-2026.

El objetivo es dejar una base confiable y enriquecida para el analisis exploratorio, el dashboard de BI y el modelo predictivo.

## 1. Preparacion del entorno

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT           = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_PATH       = ROOT / 'data' / 'raw'       / 'Cambio_climatico.csv'
CLEANED_PATH   = ROOT / 'data' / 'cleaned'   / 'enso_cleaned.csv'
PROCESSED_PATH = ROOT / 'data' / 'processed' / 'enso_model_ready.csv'


## 2. Fuente de datos

La fuente principal es `data/raw/Cambio_climatico.csv`.

Contiene registros mensuales del indice ENSO desde enero de 1950 hasta abril de 2026, incluyendo temperatura superficial del oceano Pacifico ecuatorial y anomalias respecto al promedio historico.

In [ ]:
raw_df = pd.read_csv(RAW_PATH)
display(raw_df.head())
print(f'Filas:    {raw_df.shape[0]:,}')
print(f'Columnas: {raw_df.shape[1]}')
print(f'Nombres : {list(raw_df.columns)}')

## 3. Extraccion

Se lee el CSV sin modificaciones. Los datos se presentan tal como vienen de la fuente.

In [ ]:
display(raw_df.head(10))
print(f'Filas    : {raw_df.shape[0]:,}')
print(f'Columnas : {raw_df.shape[1]}')

## 4. Diagnostico inicial

Revisamos tipos de dato, valores faltantes, duplicados y rango temporal.

In [ ]:
diagnostic_summary = pd.DataFrame({
    'tipo_dato'     : raw_df.dtypes.astype(str),
    'nulos'         : raw_df.isnull().sum(),
    'pct_nulos'     : (raw_df.isnull().mean() * 100).round(2),
    'valores_unicos': raw_df.nunique(),
})
display(diagnostic_summary)
print(f'Duplicados exactos : {raw_df.duplicated().sum()}')
print(f'Rango fecha        : {raw_df["date"].min()}  ->  {raw_df["date"].max()}')

In [ ]:
raw_df.describe(include='all').T

## 5. Transformacion

La transformacion realiza los siguientes pasos:
- Convierte `date` a formato datetime
- Renombra las columnas al espanol con nombres claros
- Estandariza las categorias de fase al espanol
- Elimina los 4 nulos existentes en columnas de trimestre

In [ ]:
cleaned_df = raw_df.copy()

# Paso 1: convertir fecha
cleaned_df['date'] = pd.to_datetime(cleaned_df['date'])

# Paso 2: renombrar columnas
COLUMN_MAP = {
    'date'                 : 'Fecha',
    'TOTAL'                : 'Temperatura_Pacifico_C',
    'ClimAdjust'           : 'Temperatura_Ajustada_C',
    'ANOM'                 : 'Anomalia_C',
    'ANOM_trimester'       : 'Anomalia_Trimestral_C',
    'ANOM_trimester_round' : 'Anomalia_Trimestral_Round',
    'phase_trimester'      : 'Fase_Trimestral',
    'phase_event'          : 'Fase_Evento',
}
cleaned_df = cleaned_df.rename(columns=COLUMN_MAP)

# Paso 3: estandarizar categorias al espanol
FASE_MAP = {'elnino': 'El Nino', 'lanina': 'La Nina', 'neutral': 'Neutral'}
cleaned_df['Fase_Trimestral'] = cleaned_df['Fase_Trimestral'].map(FASE_MAP)
cleaned_df['Fase_Evento']     = cleaned_df['Fase_Evento'].map(FASE_MAP)

# Paso 4: eliminar nulos
cleaned_df = cleaned_df.dropna().reset_index(drop=True)

print(f'Filas despues de limpieza : {len(cleaned_df):,}')
print(f'Nulos restantes           : {cleaned_df.isnull().sum().sum()}')
display(cleaned_df.head())

In [ ]:
quality_report = pd.DataFrame({
    'tipo_dato'     : cleaned_df.dtypes.astype(str),
    'nulos'         : cleaned_df.isnull().sum(),
    'valores_unicos': cleaned_df.nunique(),
})
display(quality_report)
print(f'Duplicados: {cleaned_df.duplicated().sum()}')

## 7. Variables derivadas

Se generan variables adicionales a partir de las columnas existentes:

- **Temporales**: Anio, Mes, Decada, Trimestre
- **Intensidad**: clasifica la anomalia en Neutral / Debil / Moderado / Fuerte / Muy fuerte (estandar NOAA)
- **Duracion**: meses consecutivos en la misma fase
- **Evento extremo**: bandera binaria cuando |Anomalia| >= 1.5 C
- **Tendencia 12m**: promedio movil anual de la anomalia
- **Delta**: cambio mes a mes

In [ ]:
processed_df = cleaned_df.copy()

# Variables temporales
processed_df['Anio']      = processed_df['Fecha'].dt.year
processed_df['Mes']       = processed_df['Fecha'].dt.month
processed_df['Decada']    = (processed_df['Anio'] // 10 * 10).astype(str) + 's'
processed_df['Trimestre'] = processed_df['Fecha'].dt.quarter

# Intensidad del evento (clasificacion estandar NOAA)
def clasificar_intensidad(anom, fase):
    a = abs(anom)
    if fase == 'Neutral': return 'Neutral'
    if a < 1.0:           return 'Debil'
    if a < 1.5:           return 'Moderado'
    if a < 2.0:           return 'Fuerte'
    return 'Muy fuerte'

processed_df['Intensidad_Evento'] = processed_df.apply(
    lambda r: clasificar_intensidad(r['Anomalia_C'], r['Fase_Evento']), axis=1
)

# Duracion del evento
processed_df['_cambio'] = (processed_df['Fase_Evento'] != processed_df['Fase_Evento'].shift(1)).astype(int)
processed_df['_id']     = processed_df['_cambio'].cumsum()
dur = processed_df.groupby('_id').size().reset_index(name='Duracion_Meses')
processed_df = processed_df.merge(dur, on='_id').drop(columns=['_cambio', '_id'])

# Evento extremo
processed_df['Evento_Extremo'] = (processed_df['Anomalia_C'].abs() >= 1.5).astype(int)

# Tendencia 12 meses
processed_df['Anomalia_12m']   = processed_df['Anomalia_C'].rolling(12, min_periods=6).mean().round(3)

# Delta anomalia
processed_df['Anomalia_Delta'] = processed_df['Anomalia_C'].diff().round(3)

# Reordenar columnas
cols = [
    'Fecha', 'Anio', 'Mes', 'Trimestre', 'Decada',
    'Temperatura_Pacifico_C', 'Temperatura_Ajustada_C',
    'Anomalia_C', 'Anomalia_Trimestral_C', 'Anomalia_Trimestral_Round',
    'Anomalia_12m', 'Anomalia_Delta',
    'Fase_Trimestral', 'Fase_Evento', 'Intensidad_Evento',
    'Duracion_Meses', 'Evento_Extremo',
]
processed_df = processed_df[cols].reset_index(drop=True)

print(f'Variables totales: {processed_df.shape[1]}')
display(processed_df.head(10))

## 8. Carga de resultados

Se guardan los dos datasets: limpio y procesado con variables derivadas.

In [ ]:
cleaned_df.to_csv(CLEANED_PATH, index=False)
processed_df.to_csv(PROCESSED_PATH, index=False)

print('Archivos guardados:')
print(f'  cleaned   -> enso_cleaned.csv      ({len(cleaned_df):,} filas, {cleaned_df.shape[1]} columnas)')
print(f'  processed -> enso_model_ready.csv  ({len(processed_df):,} filas, {processed_df.shape[1]} columnas)')

## 9. Resultados de la Fase 1

- Diagnostico: dataset limpio, sin duplicados, con 4 nulos eliminados.
- Transformacion: columnas renombradas al espanol, fechas convertidas, categorias estandarizadas.
- Variables derivadas: 9 nuevas variables construidas desde las columnas originales.
- Dos archivos de salida:
  - `enso_cleaned.csv` - datos limpios y renombrados (914 filas, 8 columnas)
  - `enso_model_ready.csv` - datos enriquecidos para BI y modelo (914 filas, 17 columnas)